# Structured Output 

Models can be requested to provide their response in a format matching a given schema.This is useful for ensuring the output can be easily parsed and used in subsequent processing. Langchain supports multipleschema types and methods for enforcing structured outputs.

# 1. Pydantic 

Pydantic models provide the richest feature set with field validation, descriptions and nested structures

In [3]:
import os 
from langchain.chat_models import init_chat_model
os.environ['GROQ_API_KEY'] = os.getenv('GROQ_API_KEY')
model = init_chat_model("groq:llama-3.3-70b-versatile")
model

ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x10df0d940>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x10df0e660>, model_name='llama-3.3-70b-versatile', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [ ]:
from pydantic import BaseModel, Field 

class Movie(BaseModel):
    title:str = Field(description="The title of the movie")
    year:int = Field(description="This year the movie was released")
    director:str = Field(description="The director of the movie")
    rating:float = Field(description="Movies ratings out of 10")

In [6]:
model_with_structure = model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x10df0d940>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x10df0e660>, model_name='llama-3.3-70b-versatile', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of the movie', 'type': 'string'}, 'year': {'description': 'This year the movie was released', 'type': 'integer'}, 'director': {'description': 'The director of the movie', 'type': 'string'}, 'rating': {'description': 'Movies ratings out of 10'

In [7]:
model_with_structure.invoke("Provide details about the K-Drama - True Beauty")

Movie(title='True Beauty', year=2020, director='Kim Sang-hyeop', rating=8.1)

In [8]:
# Message Output alongside parsed structure

class Movie(BaseModel):
    title:str = Field(..., description="The title of the movie")
    year:int = Field(..., description="This year the movie was released")
    director:str = Field(..., description="The director of the movie")
    rating:float = Field(..., description="Movies ratings out of 10")
model_with_structure = model.with_structured_output(Movie, include_raw=True)

model_with_structure.invoke("Provide details about the K-Drama - True Beauty")

{'raw': AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '6w8wfqewz', 'function': {'arguments': '{"director":"Kim Sang-hyeop","rating":8.1,"title":"True Beauty","year":2020}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 35, 'prompt_tokens': 281, 'total_tokens': 316, 'completion_time': 0.092243224, 'completion_tokens_details': None, 'prompt_time': 0.027959963, 'prompt_tokens_details': None, 'queue_time': 0.163229699, 'total_time': 0.120203187}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_3272ea2d91', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e72ff-cebb-7901-9d17-e444379127af-0', tool_calls=[{'name': 'Movie', 'args': {'director': 'Kim Sang-hyeop', 'rating': 8.1, 'title': 'True Beauty', 'year': 2020}, 'id': '6w8wfqewz', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 281, 'output_tokens': 35, '

In [10]:
# Nested Structure 

class Actor(BaseModel):
    name : str
    year : int
class MovieDetails(BaseModel):
    title : str 
    year : int
    cast : list[Actor]
    genres : list[str]
    budget : float | None = Field(None, description="Budget in INR")

model_with_structure = model.with_structured_output(MovieDetails)
response = model_with_structure.invoke("Provide details about the K-Drama - True Beauty")
response


MovieDetails(title='True Beauty', year=2020, cast=[Actor(name='Moon Ga-young', year=2020), Actor(name='Park Yoo-na', year=2020)], genres=['Romance', 'Comedy', 'Drama'], budget=None)

# TypedDict

TypeDict provides a simpler alternative using python's built-in typing, ideal when you don't need runtime validation. 

In [14]:
from typing_extensions import TypedDict, Annotated 

class MovieDict(TypedDict):
    """ A movie with details"""
    title : Annotated[str,...,"The title of the movie"]
    year : Annotated[int, ..., "The year the movie was released"]
    director : Annotated[str, ...,"The director of the movie"]
    rating : Annotated[float, ..., "The movie's rating out of 10"]

model_with_typedict = model.with_structured_output(MovieDict)
response = model_with_typedict.invoke("Provide details about the K-Drama - Strong girl do bong soon")
response

{'director': 'Lee Hyung-min',
 'rating': 8.1,
 'title': 'Strong Girl Do Bong Soon',
 'year': 2017}

In [15]:
class Actor(TypedDict):
    name : str
    year : int
class MovieDetails(TypedDict):
    title : str 
    year : int
    cast : list[Actor]
    genres : list[str]
    budget : float | None = Field(None, description="Budget in INR")

model_with_structure = model.with_structured_output(MovieDetails)
response = model_with_structure.invoke("Provide details about the K-Drama - True Beauty")
response


{'budget': 0,
 'cast': [{'name': 'Moon Ga-young', 'year': 2020},
  {'name': 'Park Yoo-na', 'year': 2020},
  {'name': 'Cha Eun-woo', 'year': 2020}],
 'genres': ['Romance', 'Comedy', 'Drama'],
 'title': 'True Beauty',
 'year': 2020}

In [17]:
model.profile

{'max_input_tokens': 131072,
 'max_output_tokens': 32768,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': False,
 'tool_calling': True}

# DataClasses 

A data class is a class typically containing mainly data, although there aren't really and restrictions. You create it using the @dataclass decorator

In [18]:
import os
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')

In [23]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent

class Contactinfo(BaseModel):
    """Contact information for a person"""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email of the person")
    phone: str = Field(description="The phone number of the person")

agent = create_agent(
    model="gpt-5",
    response_format=Contactinfo #Auto-selects providerstartegy
)

result = agent.invoke({
    "messages" : [{"role":"user","content":"Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result

{'messages': [HumanMessage(content='Extract contact info from: John Doe, john@example.com, (555) 123-4567', additional_kwargs={}, response_metadata={}, id='56c6140f-add1-4e88-8978-f1bd8afae6be'),
  AIMessage(content='{"name":"John Doe","email":"john@example.com","phone":"(555) 123-4567"}', additional_kwargs={'parsed': None, 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 804, 'prompt_tokens': 203, 'total_tokens': 1007, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 768, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DkqlIArBohXAaSUFbfMfTr5C4hkUB', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e73bd-c04c-71f1-b92a-ff8a73dd54ee-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 203, 'outpu

In [21]:
print(result['structured_response'])

name='John Doe' email='john@example.com' phone='(555) 123-4567'


In [25]:
## TypeDict 

from typing_extensions import TypedDict
from langchain.agents import create_agent

class Contactinfo(TypedDict):
    """contact info of a person"""
    name: str 
    email: str 
    phone: str

agent = create_agent(
    model="gpt-5",
    response_format=Contactinfo #Auto-selects providerstartegy
)

result = agent.invoke({
    "messages" : [{"role":"user","content":"Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result['structured_response']

{'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}

In [ ]:
## DataClass

from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class Contactinfo(TypedDict):
    """contact info of a person"""
    name: str 
    email: str 
    phone: str

agent = create_agent(
    model="gpt-5",
    response_format=Contactinfo #Auto-selects providerstartegy
)

result = agent.invoke({
    "messages" : [{"role":"user","content":"Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result['structured_response']